# Dependencies

In [1]:
# Install dependencies found in Pipfile
%pip install torch sentence-transformers qdrant-client psycopg2-binary tqdm pydantic python-dotenv sshtunnel

# for better output
%pip install --upgrade jupyter ipywidgets widgetsnbextension

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Configuration & SSH Tunnels

In [ ]:
import time
import paramiko
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# --- PATCH PARAMIKO ---
if not hasattr(paramiko, "DSSKey"):
    class DSSKey(paramiko.PKey):
        pass
    paramiko.DSSKey = DSSKey

# --- CONFIGURATION ---
SSH_HOST = os.getenv("SSH_HOST", "95.216.74.52")
SSH_USER = os.getenv("SSH_USER", "lean")
SSH_PASSWORD = os.getenv("SSH_PASSWORD")

# Postgres
PG_REMOTE_IP = os.getenv("PG_REMOTE_IP", "10.0.1.7")
PG_REMOTE_PORT = int(os.getenv("PG_REMOTE_PORT", "5432"))
PG_LOCAL_PORT = int(os.getenv("DB_PORT", "5433"))

# Qdrant (We will try both ports)
QDRANT_REMOTE_IP = os.getenv("QDRANT_REMOTE_IP", "10.0.3.2")

# --- START TUNNEL ---
print("Establishing consolidated SSH Tunnel...")

tunnel = SSHTunnelForwarder(
    (SSH_HOST, 22),
    ssh_username=SSH_USER,
    ssh_password=SSH_PASSWORD,
    remote_bind_addresses=[
        (PG_REMOTE_IP, PG_REMOTE_PORT),      # Postgres
        (QDRANT_REMOTE_IP, 6333),             # Qdrant HTTP
        (QDRANT_REMOTE_IP, 6334)              # Qdrant GRPC
    ],
    local_bind_addresses=[
        ('127.0.0.1', PG_LOCAL_PORT),
        ('127.0.0.1', 6333),
        ('127.0.0.1', 6334)
    ],
    set_keepalive=10.0,
    mute_exceptions=True
)

tunnel.start()
time.sleep(2) # Wait for connections

if tunnel.is_active:
    print(f"✅ Tunnel Active!")
    print(f"   - Postgres:    localhost:{PG_LOCAL_PORT} -> {PG_REMOTE_IP}:{PG_REMOTE_PORT}")
    print(f"   - Qdrant HTTP: localhost:6333 -> {QDRANT_REMOTE_IP}:6333")
    print(f"   - Qdrant GRPC: localhost:6334 -> {QDRANT_REMOTE_IP}:6334")
else:
    print("❌ Tunnel failed to start.")

def close_tunnels():
    if tunnel.is_active:
        tunnel.stop()
    print("Tunnel closed.")

Establishing consolidated SSH Tunnel...
✅ Tunnel Active!
   - Postgres:    localhost:5433 -> 10.0.1.7:5432
   - Qdrant HTTP: localhost:6333 -> 10.0.3.2:6333
   - Qdrant GRPC: localhost:6334 -> 10.0.3.2:6334


# Database Class (db.py)

In [2]:
import psycopg2
from psycopg2 import pool
from datetime import datetime
from typing import Dict, List, Optional
import threading

class Database:
    """PostgreSQL database operations with connection pooling for thread safety."""
    
    def __init__(self, config: Dict, min_connections: int = 1, max_connections: int = 10):
        # Use a thread-safe connection pool
        self.pool = pool.ThreadedConnectionPool(
            min_connections, 
            max_connections,
            **config
        )
        self._local = threading.local()
    
    def _get_conn(self):
        """Get a connection from the pool for the current thread."""
        if not hasattr(self._local, 'conn') or self._local.conn is None or self._local.conn.closed:
            self._local.conn = self.pool.getconn()
            self._local.conn.set_session(readonly=True, autocommit=True)
        return self._local.conn
    
    def _return_conn(self):
        """Return connection to pool."""
        if hasattr(self._local, 'conn') and self._local.conn is not None:
            self.pool.putconn(self._local.conn)
            self._local.conn = None
    
    def get_datasets(self) -> List[Dict]:
        """Fetch all datasets with document counts."""
        conn = self._get_conn()
        with conn.cursor() as cur:
            cur.execute("""
                SELECT 
                    d.id, d.name, d.version, d.split, d.release_date, d.source_uri,
                    COUNT(did.document_id) as doc_count
                FROM dataset d
                LEFT JOIN document_in_dataset did ON d.id = did.dataset_id
                GROUP BY d.id, d.name, d.version, d.split, d.release_date, d.source_uri
                ORDER BY d.name
            """)
            columns = [desc[0] for desc in cur.description]
            return [dict(zip(columns, row)) for row in cur.fetchall()]
    
    def get_dataset_by_name(self, name: str) -> Optional[Dict]:
        """Get dataset by name."""
        return next((ds for ds in self.get_datasets() if ds['name'].lower() == name.lower()), None)
    
    def get_document_ids(self, dataset_id: str, limit: Optional[int] = None, after_doc_id: Optional[str] = None) -> List[str]:
        """Fetch document IDs for a dataset, optionally starting after a specific doc_id."""
        conn = self._get_conn()
        with conn.cursor() as cur:
            if after_doc_id:
                query = """
                    SELECT document_id FROM document_in_dataset 
                    WHERE dataset_id = %s AND document_id > %s 
                    ORDER BY document_id
                """
                params = (dataset_id, after_doc_id)
            else:
                query = "SELECT document_id FROM document_in_dataset WHERE dataset_id = %s ORDER BY document_id"
                params = (dataset_id,)
            
            if limit:
                query += f" LIMIT {limit}"
            cur.execute(query, params)
            return [row[0] for row in cur.fetchall()]
    
    def get_documents(self, doc_ids: List[str]) -> List[Dict]:
        """Fetch full documents by IDs."""
        if not doc_ids:
            return []
        
        conn = self._get_conn()
        documents = []
        
        # Fetch all at once for small batches
        with conn.cursor() as cur:
            cur.execute("""
                SELECT id, url, title, body, length, created_at, updated_at
                FROM documents WHERE id = ANY(%s)
            """, (doc_ids,))
            
            columns = [desc[0] for desc in cur.description]
            for row in cur.fetchall():
                doc = dict(zip(columns, row))
                for k, v in doc.items():
                    if isinstance(v, datetime):
                        doc[k] = v.isoformat()
                documents.append(doc)
        
        return documents
    
    def close(self):
        """Close all connections in the pool."""
        self.pool.closeall()

# Ingestion Logic (ingest.py)

In [ ]:
import uuid
import requests
import grpc
from typing import List, Dict, Set, Optional, Tuple
import threading
import queue
import time

import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer

# --- PROGRESS BAR SETUP ---
# Force standard text-based progress bar for better compatibility in VS Code
from tqdm import tqdm

# --- PIPELINE CONFIGURATION ---
DATASET_NAME = "small"
BATCH_SIZE = 100  # Reduced from 500 for more frequent progress updates
QDRANT_UPSERT_BATCH = 1000  # Max points per upsert
PURGE_COLLECTION = False
RESUME_INGESTION = True

# Pipeline Config
PREFETCH_BATCHES = 10  # Increased buffer size since batches are smaller
NUM_EMBED_WORKERS = 1  # Number of parallel embedding workers (each loads its own model)
                       # For A100 (40-80GB VRAM), try 4-8 workers with 0.6B model (~2-3GB each)
                       # Set to 1 for consumer GPUs with limited VRAM

# DB Config - Load from environment variables
DB_CONFIG = {
    'host': os.getenv('DB_HOST', '127.0.0.1'),
    'port': int(os.getenv('DB_PORT', '5433')),
    'database': os.getenv('PG_DB', 'postgres'),
    'user': os.getenv('PG_USER', 'postgres'),
    'password': os.getenv('PG_PASS'),
    'sslmode': os.getenv('DB_SSLMODE', 'disable')
}

# Model Config
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

# --- CONNECTION CHECKER ---
def get_qdrant_client():
    """Get Qdrant client with increased timeout for SSH tunnel."""
    print("🔎 Checking Qdrant connectivity...")
    
    qdrant_host = os.getenv('QDRANT_HOST', '127.0.0.1')
    qdrant_http_port = int(os.getenv('QDRANT_HTTP_PORT', '6333'))
    qdrant_grpc_port = int(os.getenv('QDRANT_GRPC_PORT', '6334'))
    qdrant_api_key = os.getenv('QDRANT_API_KEY')
    
    try:
        requests.get(f"http://{qdrant_host}:{qdrant_http_port}/collections", timeout=5)
        print(f"✅ Detected Qdrant via HTTP (Port {qdrant_http_port})")
        return QdrantClient(
            host=qdrant_host, 
            port=qdrant_http_port, 
            prefer_grpc=False, 
            https=False,
            api_key=qdrant_api_key,
            timeout=300  # Increased timeout
        )
    except Exception:
        pass

    try:
        with grpc.insecure_channel(f'{qdrant_host}:{qdrant_grpc_port}') as channel:
            grpc.channel_ready_future(channel).result(timeout=5)
        print(f"✅ Detected Qdrant via GRPC (Port {qdrant_grpc_port})")
        return QdrantClient(
            host=qdrant_host, 
            port=qdrant_grpc_port, 
            prefer_grpc=True, 
            https=False,
            api_key=qdrant_api_key,
            timeout=300
        )
    except Exception:
        pass
        
    raise ConnectionError(f"Could not connect to Qdrant on port {qdrant_http_port} or {qdrant_grpc_port} via tunnel.")

# --- HELPER FUNCTIONS ---
def setup_qdrant(client: QdrantClient, vector_size: int, collection_name: str, purge: bool = False):
    if purge and client.collection_exists(collection_name):
        print(f"Purging existing collection '{collection_name}'...")
        client.delete_collection(collection_name=collection_name)

    if not client.collection_exists(collection_name):
        print(f"Creating collection '{collection_name}' with vector size {vector_size}...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
            # Optimize for batch ingestion
            optimizers_config=models.OptimizersConfigDiff(
                indexing_threshold=50000,  # Delay indexing during bulk insert
            )
        )
    else:
        print(f"Collection '{collection_name}' ready (existing).")

def get_last_processed_doc_id(qdrant: QdrantClient, collection_name: str, dataset_name: str) -> Optional[str]:
    """Get the maximum doc_id that has been processed for a given dataset."""
    try:
        if not qdrant.collection_exists(collection_name):
            return None
        
        collection_info = qdrant.get_collection(collection_name)
        if collection_info.points_count == 0:
            return None
        
        all_doc_ids: Set[str] = set()
        offset = None
        
        print("📊 Fetching existing doc_ids from Qdrant to find resume point...")
        
        while True:
            scroll_result = qdrant.scroll(
                collection_name=collection_name,
                scroll_filter=models.Filter(
                    must=[models.FieldCondition(key="dataset", match=models.MatchValue(value=dataset_name))]
                ),
                limit=10000,
                offset=offset,
                with_payload=["doc_id"],
                with_vectors=False
            )
            
            points, next_offset = scroll_result
            if not points:
                break
            for point in points:
                if point.payload and "doc_id" in point.payload:
                    all_doc_ids.add(point.payload["doc_id"])
            if next_offset is None:
                break
            offset = next_offset
        
        if not all_doc_ids:
            return None
        
        max_doc_id = max(all_doc_ids)
        print(f"✅ Found {len(all_doc_ids)} unique documents already processed. Last doc_id: {max_doc_id}")
        return max_doc_id
        
    except Exception as e:
        print(f"⚠️ Could not determine last processed doc_id: {e}")
        return None

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    """
    Faster chunking by character length approximation.
    Assumes ~3 characters per token. 512 tokens ~= 1500 chars.
    """
    if not text: return []
    
    # Approx char count for 512 tokens
    CHAR_CHUNK = chunk_size * 3 
    CHAR_OVERLAP = overlap * 3
    
    if len(text) <= CHAR_CHUNK: return [text]
    
    chunks = []
    stride = CHAR_CHUNK - CHAR_OVERLAP
    for i in range(0, len(text), stride):
        chunks.append(text[i : i + CHAR_CHUNK])
    return chunks

def process_documents_to_chunks(docs: List[Dict], dataset_name: str) -> Tuple[List[str], List[Dict]]:
    """
    Pre-process documents into chunks and metadata.
    Returns: (list of chunk texts, list of metadata dicts)
    """
    all_texts = []
    all_metadata = []
    
    for doc in docs:
        doc_id = doc['id']
        text = doc.get('body', '') or doc.get('title', '')
        if not text:
            continue
        
        chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        
        for chunk_idx, chunk_text_content in enumerate(chunks):
            all_texts.append(chunk_text_content)
            all_metadata.append({
                "doc_id": doc_id,
                "chunk_index": chunk_idx,
                "text": chunk_text_content,
                "dataset": dataset_name,
                "original_title": doc.get('title', ''),
                "url": doc.get('url', '')
            })
    
    return all_texts, all_metadata

def create_points_from_embeddings(embeddings, metadata_list: List[Dict]) -> List:
    """Create Qdrant points from embeddings and metadata."""
    points = []
    for embedding, meta in zip(embeddings, metadata_list):
        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{meta['doc_id']}_chunk_{meta['chunk_index']}"))
        points.append(models.PointStruct(
            id=point_id, 
            vector=embedding.tolist(), 
            payload=meta
        ))
    return points

class PipelinedIngestion:
    """
    High-performance ingestion using pipelining:
    - Stage 1: Prefetch documents from DB (I/O bound)
    - Stage 2: Chunk and embed documents (GPU bound) - supports multiple parallel workers
    - Stage 3: Upsert to Qdrant (Network I/O bound)
    
    For large GPUs (A100 40-80GB), multiple model instances can be loaded
    to maximize GPU utilization. Each embed worker has its own model.
    """
    
    def __init__(self, db, qdrant, models_list: List, dataset_name: str, collection_name: str,
                 prefetch_size: int = 4):
        self.db = db
        self.qdrant = qdrant
        self.models = models_list  # List of model instances for parallel embedding
        self.num_embed_workers = len(models_list)
        self.dataset_name = dataset_name
        self.collection_name = collection_name
        self.prefetch_size = prefetch_size
        
        # Queues for pipeline stages - increase size for multi-worker
        queue_size = prefetch_size * max(1, self.num_embed_workers)
        self.doc_queue = queue.Queue(maxsize=queue_size)
        self.points_queue = queue.Queue(maxsize=queue_size)
        
        # Synchronization
        self.stop_event = threading.Event()
        self.error_event = threading.Event()
        self.error_message = None
        self.active_embed_workers = self.num_embed_workers  # Track active workers
        self.workers_lock = threading.Lock()
        
        # Stats
        self.stats_lock = threading.Lock()
        self.total_docs = 0
        self.total_chunks = 0
        
        # Per-stage timing stats (for bottleneck analysis)
        self.prefetch_times = []  # Time per batch fetch
        self.embed_times = []     # Time per batch embed
        self.upsert_times = []    # Time per batch upsert
        self.prefetch_docs = 0    # Docs fetched
        self.embed_docs = 0       # Docs embedded
        self.upsert_docs = 0      # Docs upserted
    
    def _prefetch_worker(self, batches: List[List[str]]):
        """Worker that prefetches document batches from DB."""
        try:
            for batch_idx, batch_ids in enumerate(batches):
                if self.stop_event.is_set() or self.error_event.is_set():
                    break
                
                start_time = time.time()
                docs = self.db.get_documents(batch_ids)
                elapsed = time.time() - start_time
                
                with self.stats_lock:
                    self.prefetch_times.append(elapsed)
                    self.prefetch_docs += len(batch_ids)
                
                self.doc_queue.put((batch_idx, batch_ids, docs))
            
            # Signal end of documents - one sentinel per embed worker
            for _ in range(self.num_embed_workers):
                self.doc_queue.put(None)
        except Exception as e:
            self.error_event.set()
            self.error_message = f"Prefetch error: {e}"
            for _ in range(self.num_embed_workers):
                self.doc_queue.put(None)
    
    def _embed_worker(self, worker_id: int, model):
        """Worker that processes documents and generates embeddings."""
        batches_processed = 0
        try:
            while not self.stop_event.is_set() and not self.error_event.is_set():
                try:
                    item = self.doc_queue.get(timeout=30)
                except queue.Empty:
                    continue
                
                if item is None:
                    break
                
                batch_idx, batch_ids, docs = item
                start_time = time.time()
                
                # 1. CPU WORK: Chunking (Use your Fast Character Chunker)
                texts, metadata = process_documents_to_chunks(docs, self.dataset_name)
                
                if not texts:
                    self.points_queue.put((batch_idx, len(batch_ids), [], 0))
                    continue
                
                # 2. GPU WORK: Embedding
                # We start with 256. If 256 fails even with BF16, the code will auto-downgrade.
                current_batch_size = 256 
                
                try:
                    embeddings = model.encode(
                        texts, 
                        batch_size=current_batch_size,
                        show_progress_bar=False, 
                        convert_to_numpy=True,
                        normalize_embeddings=True
                        # Note: SentenceTransformer usually handles truncation via the model config we set earlier
                    )
                except torch.cuda.OutOfMemoryError:
                    print(f"⚠️ OOM with batch {current_batch_size}. Retrying with 128...")
                    torch.cuda.empty_cache()
                    try:
                        embeddings = model.encode(
                            texts, 
                            batch_size=128, 
                            show_progress_bar=False, 
                            convert_to_numpy=True,
                            normalize_embeddings=True
                        )
                    except torch.cuda.OutOfMemoryError:
                        print(f"⚠️ OOM with batch 128. Retrying with 32...")
                        torch.cuda.empty_cache()
                        embeddings = model.encode(
                            texts, 
                            batch_size=32, 
                            show_progress_bar=False, 
                            convert_to_numpy=True,
                            normalize_embeddings=True
                        )
                
                # 3. CPU WORK: Point Creation
                points = create_points_from_embeddings(embeddings, metadata)
                
                elapsed = time.time() - start_time
                with self.stats_lock:
                    self.embed_times.append(elapsed)
                    self.embed_docs += len(batch_ids)
                
                self.points_queue.put((batch_idx, len(batch_ids), points, len(texts)))
                batches_processed += 1
                
        except Exception as e:
            self.error_event.set()
            self.error_message = f"Embed worker {worker_id} error: {e}"
        finally:
            with self.workers_lock:
                self.active_embed_workers -= 1
                if self.active_embed_workers == 0:
                    self.points_queue.put(None)
    
    def _get_stage_speeds(self) -> Dict[str, str]:
        """Calculate docs/sec for each pipeline stage (rolling average of last 10 batches)."""
        speeds = {}
        window = 10  # Use last N batches for rolling average
        
        with self.stats_lock:
            # Prefetch speed
            if self.prefetch_times:
                recent = self.prefetch_times[-window:]
                avg_time = sum(recent) / len(recent)
                docs_per_sec = BATCH_SIZE / avg_time if avg_time > 0 else 0
                speeds['fetch'] = f"{docs_per_sec:.0f}"
            else:
                speeds['fetch'] = "..."
            
            # Embed speed
            if self.embed_times:
                recent = self.embed_times[-window:]
                avg_time = sum(recent) / len(recent)
                docs_per_sec = BATCH_SIZE / avg_time if avg_time > 0 else 0
                speeds['embed'] = f"{docs_per_sec:.0f}"
            else:
                speeds['embed'] = "..."
            
            # Upsert speed
            if self.upsert_times:
                recent = self.upsert_times[-window:]
                avg_time = sum(recent) / len(recent)
                docs_per_sec = BATCH_SIZE / avg_time if avg_time > 0 else 0
                speeds['upsert'] = f"{docs_per_sec:.0f}"
            else:
                speeds['upsert'] = "..."
        
        return speeds
    
    def _upsert_worker(self, pbar, total_batches: int):
        """Worker that upserts points to Qdrant."""
        completed_batches = 0
        
        try:
            while not self.stop_event.is_set():
                try:
                    item = self.points_queue.get(timeout=60)  # 60s timeout
                except queue.Empty:
                    if self.error_event.is_set():
                        break
                    continue  # Check stop_event and try again
                    
                if item is None:
                    break
                
                batch_idx, doc_count, points, chunk_count = item
                start_time = time.time()
                
                if points:
                    # Upsert in sub-batches to avoid timeout
                    for i in range(0, len(points), QDRANT_UPSERT_BATCH):
                        sub_batch = points[i:i + QDRANT_UPSERT_BATCH]
                        self.qdrant.upsert(
                            collection_name=self.collection_name, 
                            points=sub_batch,
                            wait=False  # Don't wait for indexing
                        )
                
                elapsed = time.time() - start_time
                
                with self.stats_lock:
                    self.total_docs += doc_count
                    self.total_chunks += chunk_count
                    self.upsert_times.append(elapsed)
                    self.upsert_docs += doc_count
                
                completed_batches += 1
                
                # Get per-stage speeds for progress bar
                speeds = self._get_stage_speeds()
                
                pbar.update(doc_count)
                pbar.set_postfix(
                    chunks=self.total_chunks,
                    fetch=f"{speeds['fetch']}d/s",
                    embed=f"{speeds['embed']}d/s", 
                    upsert=f"{speeds['upsert']}d/s",
                    batch=f"{completed_batches}/{total_batches}",
                    refresh=True
                )
                pbar.refresh()
                
        except Exception as e:
            self.error_event.set()
            self.error_message = f"Upsert error: {e}"
    
    def run(self, doc_ids: List[str]) -> Tuple[int, int]:
        """Run the pipelined ingestion."""
        # Prepare batches
        batches = []
        for i in range(0, len(doc_ids), BATCH_SIZE):
            batches.append(doc_ids[i:i + BATCH_SIZE])
        
        total_batches = len(batches)
        print(f"📦 Prepared {total_batches} batches of ~{BATCH_SIZE} docs each")
        print(f"🔧 Pipeline: prefetch={self.prefetch_size}, embed_workers={self.num_embed_workers}, embed_batch=64, upsert_batch={QDRANT_UPSERT_BATCH}")
        
        # Create progress bar
        pbar = tqdm(total=len(doc_ids), unit="docs", desc="Ingesting", 
                    mininterval=0.5, dynamic_ncols=True)
        
        # Start pipeline threads
        prefetch_thread = threading.Thread(target=self._prefetch_worker, args=(batches,))
        
        # Start multiple embed workers, each with its own model
        embed_threads = []
        for i, model in enumerate(self.models):
            t = threading.Thread(target=self._embed_worker, args=(i, model))
            embed_threads.append(t)
        
        upsert_thread = threading.Thread(target=self._upsert_worker, args=(pbar, total_batches))
        
        prefetch_thread.start()
        for t in embed_threads:
            t.start()
        upsert_thread.start()
        
        # Wait for completion
        prefetch_thread.join()
        for t in embed_threads:
            t.join()
        upsert_thread.join()
        
        pbar.close()
        
        if self.error_event.is_set():
            print(f"❌ Pipeline error: {self.error_message}")
        
        return self.total_chunks, self.total_docs


def run_ingestion(doc_ids: List[str], db, qdrant, dataset_name: str, 
                  collection_name: str, models_list: List) -> Tuple[int, int]:
    """Run optimized pipelined ingestion.
    
    Args:
        doc_ids: List of document IDs to process
        db: Database connection
        qdrant: Qdrant client
        dataset_name: Name of the dataset
        collection_name: Qdrant collection name
        models_list: List of SentenceTransformer model instances
    """
    
    if len(doc_ids) == 0:
        return 0, 0
    
    print(f"🚀 Starting pipelined ingestion with {len(models_list)} embed worker(s)...")
    
    pipeline = PipelinedIngestion(
        db=db,
        qdrant=qdrant,
        models_list=models_list,
        dataset_name=dataset_name,
        collection_name=collection_name,
        prefetch_size=PREFETCH_BATCHES
    )
    
    return pipeline.run(doc_ids)


In [ ]:
# --- MAIN EXECUTION ---
print(f"Initializing ingestion pipeline on device: {DEVICE}")
print(f"Config: BATCH_SIZE={BATCH_SIZE}, PREFETCH={PREFETCH_BATCHES}, EMBED_WORKERS={NUM_EMBED_WORKERS}, UPSERT_BATCH={QDRANT_UPSERT_BATCH}")

# 1. Connect to DB with connection pool
try:
    db = Database(DB_CONFIG, min_connections=2, max_connections=5)
    print("Connected to PostgreSQL (pooled).")
except Exception as e:
    print(f"Failed to connect to DB: {e}")
    raise e

# 2. Connect to Qdrant
try:
    qdrant = get_qdrant_client()
except Exception as e:
    print(f"FATAL: {e}")
    db.close()
    raise e

# 3. Load Model(s)
print(f"Loading {NUM_EMBED_WORKERS} embedding model instance(s): {EMBEDDING_MODEL_NAME}...")
models_list = []
for i in range(NUM_EMBED_WORKERS):
    print(f"   Loading model {i+1}/{NUM_EMBED_WORKERS}...")
    
    # Load model
    model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE, trust_remote_code=True)
    
    # --- FIX 1: Hard Clamp Context Window ---
    # We must tell the tokenizer NOT to prepare for 32k tokens
    model.max_seq_length = 512
    model.tokenizer.model_max_length = 512
    model.tokenizer.truncation = True
    
    # --- FIX 2: Use Half Precision (BF16) ---
    # RTX 5070 Ti supports bfloat16 (better stability than float16)
    # This cuts memory usage nearly in half
    if DEVICE == "cuda":
        model.to(torch.bfloat16) 
    
    models_list.append(model)
    
    if i == 0:
        vector_size = model.get_sentence_embedding_dimension()

# Warmup
print(f"🔥 Warming up {NUM_EMBED_WORKERS} model(s)...")
for i, model in enumerate(models_list):
    # Pass a dummy input to force CUDA initialization
    _ = model.encode(["warmup"], show_progress_bar=False, batch_size=1)

print(f"✅ Loaded {NUM_EMBED_WORKERS} model(s). Precision: BF16/FP16. Context: 512.")
if DEVICE == "cuda":
    mem_allocated = torch.cuda.memory_allocated() / (1024**3)
    mem_reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"   GPU Memory: {mem_allocated:.2f} GB allocated, {mem_reserved:.2f} GB reserved")

# 4. Select Dataset
dataset = db.get_dataset_by_name(DATASET_NAME)
if not dataset:
    print(f"Dataset '{DATASET_NAME}' not found.")
else:
    print(f"Processing dataset: {dataset['name']} ({dataset['doc_count']} total docs)")
    
    # 5. Setup Collection
    setup_qdrant(qdrant, vector_size, DATASET_NAME, purge=PURGE_COLLECTION)

    # 6. Determine resume point
    last_processed_id = None
    if RESUME_INGESTION and not PURGE_COLLECTION:
        last_processed_id = get_last_processed_doc_id(qdrant, DATASET_NAME, dataset['name'])
    
    # 7. Fetch only NEW document IDs
    if last_processed_id:
        print(f"📍 Resuming from doc_id > '{last_processed_id}'")
        doc_ids = db.get_document_ids(dataset['id'], after_doc_id=last_processed_id)
        print(f"Found {len(doc_ids)} NEW documents to process")
    else:
        print("📍 Starting fresh ingestion")
        doc_ids = db.get_document_ids(dataset['id'])
        print(f"Found {len(doc_ids)} documents to process")

    if not doc_ids:
        print("✅ All documents already ingested. Nothing to do!")
    else:
        start_time = time.time()
        
        try:
            total_chunks, total_docs = run_ingestion(
                doc_ids=doc_ids,
                db=db,
                qdrant=qdrant,
                dataset_name=dataset['name'],
                collection_name=DATASET_NAME,
                models_list=models_list
            )
            
            elapsed = time.time() - start_time
            docs_per_sec = total_docs / elapsed if elapsed > 0 else 0
            chunks_per_sec = total_chunks / elapsed if elapsed > 0 else 0
            
            print(f"\n✅ Ingestion complete!")
            print(f"   📊 {total_docs:,} documents, {total_chunks:,} chunks")
            print(f"   ⏱️  {elapsed:.1f}s ({docs_per_sec:.1f} docs/s, {chunks_per_sec:.1f} chunks/s)")
            
        except Exception as e:
            print(f"❌ Ingestion failed: {e}")
            raise

db.close()
print("Database connection closed.")

In [ ]:
from transformers import AutoTokenizer
import numpy as np


def estimate_token_distribution(sample_size=2000):
    print(f"📊 Estimating token distribution for dataset: {DATASET_NAME}")
    
    try:
        # Connect to DB
        db_est = Database(DB_CONFIG)
        
        # Get Dataset info
        dataset = db_est.get_dataset_by_name(DATASET_NAME)
        if not dataset:
            print(f"Dataset {DATASET_NAME} not found")
            return

        # Fetch sample docs
        print(f"Fetching {sample_size} documents...")
        doc_ids = db_est.get_document_ids(dataset['id'], limit=sample_size)
        docs = db_est.get_documents(doc_ids)
        
        # Load Tokenizer
        print(f"Loading tokenizer: {EMBEDDING_MODEL_NAME}")
        tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        
        # Count tokens
        lengths = []
        for doc in tqdm(docs, desc="Counting tokens"):
            text = doc.get('body', '') or doc.get('title', '')
            if text:
                tokens = tokenizer.encode(text, add_special_tokens=False)
                lengths.append(len(tokens))
        
        # Stats
        if lengths:
            lengths = np.array(lengths)
            print(f"\n📈 Token Statistics (n={len(lengths)}):")
            print(f"   Mean:   {np.mean(lengths):.1f}")
            print(f"   Median: {np.median(lengths):.1f}")
            print(f"   Min:    {np.min(lengths)}")
            print(f"   Max:    {np.max(lengths)}")
            print(f"   P90:    {np.percentile(lengths, 90):.1f}")
            print(f"   P95:    {np.percentile(lengths, 95):.1f}")
            print(f"   P99:    {np.percentile(lengths, 99):.1f}")
            
            # Simple ASCII Histogram
            print("\nDistribution:")
            hist, bin_edges = np.histogram(lengths, bins=10)
            max_h = max(hist)
            for i, count in enumerate(hist):
                bar = "█" * int(20 * count / max_h)
                range_str = f"{int(bin_edges[i])}-{int(bin_edges[i+1])}"
                print(f"   {range_str:<15} | {count:>4} {bar}")
                
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if 'db_est' in locals():
            db_est.close()

estimate_token_distribution()

📊 Estimating token distribution for dataset: small
Fetching 2000 documents...
Fetching 2000 documents...
Loading tokenizer: Qwen/Qwen3-Embedding-8B
Loading tokenizer: Qwen/Qwen3-Embedding-8B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Counting tokens: 100%|██████████| 2000/2000 [00:03<00:00, 512.98it/s]


📈 Token Statistics (n=2000):
   Mean:   1701.1
   Median: 1107.5
   Min:    8
   Max:    72592
   P90:    2930.9
   P95:    4630.3
   P99:    13479.9

Distribution:
   8-7266          | 1955 ████████████████████
   7266-14524      |   25 
   14524-21783     |   12 
   21783-29041     |    4 
   29041-36300     |    1 
   36300-43558     |    1 
   43558-50816     |    1 
   50816-58075     |    0 
   58075-65333     |    0 
   65333-72592     |    1 


: 

In [4]:
def delete_specific_collection(collection_name: str):
    """
    Utility to delete a specific Qdrant collection.
    """
    print(f"🗑️ Attempting to delete collection: '{collection_name}'")
    
    try:
        # Reuse existing client if available, otherwise try to connect
        client = qdrant if 'qdrant' in locals() else get_qdrant_client()
        
        if client.collection_exists(collection_name):
            client.delete_collection(collection_name)
            print(f"✅ Collection '{collection_name}' successfully deleted.")
        else:
            print(f"⚠️ Collection '{collection_name}' not found.")
            
    except Exception as e:
        print(f"❌ Failed to delete collection: {e}")

# Example usage (uncomment to run):
# delete_specific_collection("small")
delete_specific_collection("qrels")

🗑️ Attempting to delete collection: 'qrels'
🔎 Checking Qdrant connectivity...
✅ Detected Qdrant via HTTP (Port 6333)
✅ Collection 'qrels' successfully deleted.


In [4]:
import json
from pathlib import Path

def fetch_and_save_document_ids(collection_name: str, output_file: str = None):
    """
    Fetch all unique document IDs (from payload metadata) from a Qdrant collection and save to JSON.
    
    Args:
        collection_name: Name of the Qdrant collection to query
        output_file: Output JSON file path. Defaults to '{collection_name}_document_ids.json'
    """
    if output_file is None:
        output_file = f"{collection_name}_document_ids.json"
    
    print(f"📥 Fetching document IDs from collection: '{collection_name}'")
    
    try:
        # Get Qdrant client
        client = qdrant if 'qdrant' in dir() else get_qdrant_client()
        
        if not client.collection_exists(collection_name):
            print(f"❌ Collection '{collection_name}' does not exist.")
            return
        
        # Get collection info
        collection_info = client.get_collection(collection_name)
        total_points = collection_info.points_count
        print(f"📊 Collection has {total_points} points")
        
        # Scroll through all points and collect unique doc_ids from payload
        document_ids = set()
        next_offset = None
        batch_size = 10000
        points_processed = 0
        
        with tqdm(total=total_points, desc="Fetching doc_ids") as pbar:
            while True:
                records, next_offset = client.scroll(
                    collection_name=collection_name,
                    limit=batch_size,
                    offset=next_offset,
                    with_payload=['doc_id'],  # Only fetch doc_id from payload
                    with_vectors=False
                )
                
                if not records:
                    break
                
                # Extract doc_ids from payload
                for record in records:
                    if record.payload and 'doc_id' in record.payload:
                        document_ids.add(str(record.payload['doc_id']))
                
                points_processed += len(records)
                pbar.update(len(records))
                
                if next_offset is None:
                    break
        
        # Convert set to sorted list for consistent output
        document_ids_list = sorted(list(document_ids))
        
        # Save to JSON
        output_path = Path(output_file)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump({
                "collection_name": collection_name,
                "total_points": total_points,
                "unique_documents": len(document_ids_list),
                "document_ids": document_ids_list
            }, f, indent=2)
        
        print(f"✅ Saved {len(document_ids_list)} unique document IDs to '{output_path.absolute()}'")
        print(f"   (from {total_points} total points/chunks)")
        return document_ids_list
        
    except Exception as e:
        print(f"❌ Error fetching document IDs: {e}")
        return None

# Example usage (uncomment and modify collection name):
# fetch_and_save_document_ids("small")
fetch_and_save_document_ids("small_openrouter_nc", "small_openrouter_nc_doc_ids.json")

📥 Fetching document IDs from collection: 'small_openrouter_nc'
🔎 Checking Qdrant connectivity...
✅ Detected Qdrant via HTTP (Port 6333)
📊 Collection has 278350 points
📊 Collection has 278350 points


Fetching doc_ids: 100%|██████████| 278350/278350 [00:06<00:00, 41239.47it/s]

✅ Saved 278350 unique document IDs to 'c:\Users\leanf\projects\uni\RAG-with-LLMs-TREC-2025-Information-Retrieval\backend\embeddings\small_openrouter_nc_doc_ids.json'
   (from 278350 total points/chunks)


['msmarco_v2.1_doc_00_1000011624',
 'msmarco_v2.1_doc_00_1000075357',
 'msmarco_v2.1_doc_00_1000145922',
 'msmarco_v2.1_doc_00_100016623',
 'msmarco_v2.1_doc_00_1000211763',
 'msmarco_v2.1_doc_00_1000235388',
 'msmarco_v2.1_doc_00_1000300778',
 'msmarco_v2.1_doc_00_1000359351',
 'msmarco_v2.1_doc_00_1000387116',
 'msmarco_v2.1_doc_00_1000490781',
 'msmarco_v2.1_doc_00_1000802318',
 'msmarco_v2.1_doc_00_1000809976',
 'msmarco_v2.1_doc_00_1000869046',
 'msmarco_v2.1_doc_00_1000881548',
 'msmarco_v2.1_doc_00_1000892843',
 'msmarco_v2.1_doc_00_1000963474',
 'msmarco_v2.1_doc_00_1001017093',
 'msmarco_v2.1_doc_00_1001041389',
 'msmarco_v2.1_doc_00_1001051509',
 'msmarco_v2.1_doc_00_1001089181',
 'msmarco_v2.1_doc_00_1001150584',
 'msmarco_v2.1_doc_00_1001190055',
 'msmarco_v2.1_doc_00_1001274032',
 'msmarco_v2.1_doc_00_1001351092',
 'msmarco_v2.1_doc_00_1001371295',
 'msmarco_v2.1_doc_00_1001427708',
 'msmarco_v2.1_doc_00_1001540718',
 'msmarco_v2.1_doc_00_1001629428',
 'msmarco_v2.1_doc_00

2025-11-30 17:49:17,381| ERROR   | Socket exception: An existing connection was forcibly closed by the remote host (10054)


# Cleanup 

In [ ]:
close_tunnels()

Tunnel closed.
